# CME Futures: Strategy Analysis

The four preceding notebooks each produced a registered, complete population of validation
backtests: the equal-weight signal baseline over every model configuration and checkpoint, the
alternative allocators over the signal shortlist, the transaction-cost grid, and the position-risk
overlays. This notebook reads those registered results and names one case-study configuration
from them.

A case-study configuration is a model configuration - family, settings, and the training
checkpoint the predictions came from - together with the signal, sizing, and risk rules applied to
it. The configuration with the highest validation Sharpe across the signal, allocation, and
risk-overlay stages is the one selected. Cost-sensitivity backtests vary the friction assumption
on a configuration already chosen, so they describe it rather than compete with it and are not in
the pool.

Both return horizons are selected from together. The horizon is part of the configuration, so it
is read off the row that wins rather than fixed before looking.

The holdout period is a later date range that no notebook up to this point has touched. It is
evaluated on the selected configuration alone, one time, and it may disagree with the validation
result. That disagreement is an outcome to report, not a reason to select again.

Prerequisites: `13_backtest`, `14_portfolio_management`, `15_risk_management`, and `16_costs`.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# CME 期貨：策略分析

前四份 Notebook 各自產生一個已登錄且完整的驗證期回測群體：涵蓋每種模型設定與檢查點的等權重訊號基準、套用於訊號候選名單的其他配置方法、交易成本網格，以及部位風險覆蓋層。本 Notebook 讀取這些已登錄結果，並從中指定一項案例研究設定。

案例研究設定包含模型設定——模型家族、參數，以及產生預測時所用的訓練檢查點——和套用到它的訊號、部位規模與風險規則。訊號、配置與風險覆蓋層階段中，驗證期夏普比率最高的設定會被選中。成本敏感度回測是在已選設定上改變交易摩擦假設，因此是描述該設定，而不是與其他設定競爭，不納入候選池。

兩個報酬率預測期間會一起參與選擇。期間是設定的一部分，因此會從勝出的資料列讀取，而非事先固定。

保留樣本是較晚的一段日期範圍，先前沒有任何 Notebook 使用過。它只會針對選定設定評估一次，結果可能與驗證期不同。這種差異是應報告的結果，不是重新選擇的理由。

先決條件：`13_backtest`、`14_portfolio_management`、`15_risk_management` 與 `16_costs`。
<!-- ML4T_ZH_END c0000-p000 -->


In [ ]:
"""Select and describe one CME futures case-study configuration."""

import plotly.express as px
import polars as pl

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    final_selection_candidate_set,
    final_validation_candidate_set,
    final_validation_results,
    open_study,
    product_universe_table,
    selection_catalog,
)
from case_studies.research import OfficialPopulation, Result
from case_studies.utils.cohort_metrics import compute_and_register
from case_studies.utils.paired_metrics import populate_paired_metrics
from case_studies.utils.strategy_analysis import (
    resolve_solvent_carrier,
    select_holdout_self_backtest,
)
from case_studies.utils.uncertainty import ENTIRE_REGISTRY
from utils.style import COLORS

In [ ]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None
PREVIEW_LABELS: list[str] = []

# The per-label candidate sets this notebook freezes are immutable under their names too, and
# for the same reason as the population above: `CandidateSet.create` refuses a changed member
# list under a name that already exists. Nothing reached that argument before, so any run whose
# membership moved - which a wider sweep does by construction - stopped at the freeze after the
# fit, with no parameter able to answer it.
#
# Each name maps to the generation this run retires. `"live"` names the lineage and looks the
# generation up, which is the form that does not decay: naming the head instead is correct only
# until the next publish, because `create` accepts the head and nothing else. The declaration is
# resolved through `candidate_set_supersedes` rather than offered straight, so a reader's clean
# clone - which has no generation to replace, and often no `candidate_sets` table at all -
# publishes generation one instead of being refused. An unchanged re-run never reads it: a set's
# hash is computed from its members and its contract, so the existing name binding answers.
SUPERSEDES_CANDIDATE_SETS: dict[str, str] = {
    "cme_futures-pre-overlay-fwd_ret_5d-v1": "live",
    "cme_futures-pre-overlay-fwd_ret_21d-v1": "live",
    "cme_futures-final-validation-fwd_ret_5d-v1": "live",
    "cme_futures-final-validation-fwd_ret_21d-v1": "live",
    "cme_futures-final-selection-v1": "live",
}

## The pool the configuration is selected from

Each per-label pool opens the three stage populations and fails if any member is missing,
incomplete, or produced under a preview identity. Combining the two horizons into one immutable
set records exactly which results were compared, so the selection can be repeated later against
the same members rather than against whatever the registry holds at the time.

<!-- ML4T_ZH_BEGIN c0003-p000 -->
## 設定選擇所依據的候選池

每個標籤的候選池會開啟三個階段的群體；若有成員缺失、不完整，或使用預覽識別資訊產生，就會失敗。把兩個預測期間合併為單一不可變集合，能精確記錄比較過哪些結果，使日後能根據相同成員重做選擇，而非依當時登錄資料庫碰巧存有的結果重新選擇。
<!-- ML4T_ZH_END c0003-p000 -->


In [ ]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
if EXECUTION_TIER == "canonical":
    if PREVIEW_LABELS:
        raise ValueError("canonical execution cannot declare preview reductions")
    labels = ALL_LABELS
elif EXECUTION_TIER == "preview":
    if WORKSPACE is None or not PREVIEW_LABELS:
        raise ValueError("preview execution requires WORKSPACE and PREVIEW_LABELS")
    unknown = sorted(set(PREVIEW_LABELS) - set(ALL_LABELS))
    if unknown:
        raise ValueError(f"preview labels this case study does not declare: {unknown}")
    labels = tuple(PREVIEW_LABELS)
else:
    raise ValueError(f"unsupported execution tier: {EXECUTION_TIER!r}")
universe = product_universe_table()
universe

Only a canonical pool is an immutable set. A preview run publishes no candidate set - one cannot
hold a preview member - so its pool is the rows its own reduced execution produced and the
`candidate_set_hash` column below is null. Everything downstream, the ranking rule included, is
the same either way; what differs is whether the pool can be reopened later by name.

<!-- ML4T_ZH_BEGIN c0005-p000 -->
只有正式候選池才是不可變集合。預覽執行不會發布候選集合——正式集合不能納入預覽成員——因此其候選池就是該次縮減版執行產生的資料列，下方的 `candidate_set_hash` 欄位為 null。無論哪種情況，後續流程及排序規則都相同；差別只在於日後能否依名稱重新開啟該候選池。
<!-- ML4T_ZH_END c0005-p000 -->


In [ ]:
if EXECUTION_TIER == "canonical":
    per_label = {
        label: final_validation_candidate_set(
            study, label=label, supersedes_by_set=SUPERSEDES_CANDIDATE_SETS
        )
        for label in labels
    }
    per_label_results = {
        label: tuple(Result.open(study, value) for value in pool_set.members)
        for label, pool_set in per_label.items()
    }
    candidates = final_selection_candidate_set(study, supersedes_by_set=SUPERSEDES_CANDIDATE_SETS)
    pool_results = tuple(Result.open(study, value) for value in candidates.members)
    pool_identity = candidates.hash
    per_label_identity = {label: pool_set.hash for label, pool_set in per_label.items()}
else:
    per_label_results = {
        label: final_validation_results(study, label=label, execution_tier=EXECUTION_TIER)
        for label in labels
    }
    pool_results = tuple(result for results in per_label_results.values() for result in results)
    pool_identity = None
    per_label_identity = dict.fromkeys(labels)
pool = selection_catalog(study, (result.hash for result in pool_results))
pool_size = pl.DataFrame(
    [
        {
            "label": label,
            "candidates": len(results),
            "candidate_set_hash": per_label_identity[label],
        }
        for label, results in per_label_results.items()
    ],
    schema={"label": pl.String, "candidates": pl.Int64, "candidate_set_hash": pl.String},
).sort("label")

## The selection correction, and the paired comparisons

Two registry tables carry the statistics this notebook reports rather than recomputes:
`cohort_metrics` holds the deflated Sharpe for each cohort's leader, and
`backtest_paired_metrics` holds bootstrap comparisons between registered return series.
Both were empty here until 2026-08-31, so an earlier edition of the README quoted deflation
numbers with nothing behind them. The cause was not a bad computation - it was that this
notebook never called for one, while `etfs`, `fx_pairs` and `us_firm_characteristics` all do.

`compute_and_register` refreshes the whole table rather than one row, so it can never report a
stale leader. `populate_paired_metrics` writes one row per comparison kind, including
`val_rank1_self` - the selected configuration's validation series against its own holdout replay,
which is the paired form of the val-to-holdout question and the only honest way to ask it.
Comparing two point estimates is not that question: the holdout is a shorter window, so the
difference carries sampling error the point estimates do not show.

The selected configuration is resolved here rather than further down because
`populate_paired_metrics` needs it. Omitting it does not fail - it falls back to ranking the
registry on raw Sharpe, which on this registry names `latent_factors`/`sdf` on `fwd_ret_21d`,
while the canonical resolver names `gbm`/`leaves_31_mse` on `fwd_ret_5d`. The paired rows would
then compare a strategy the chapter does not report, under headings that say they describe the
one it does. That is the same disagreement documented below for the holdout lookup, reaching a
different table.


<!-- ML4T_ZH_BEGIN c0007-p000 -->
## 選擇偏誤修正與配對比較

本 Notebook 報告的統計值取自兩張登錄資料庫表格，而非在此重新計算：`cohort_metrics` 儲存各群組領先者的折減夏普比率，`backtest_paired_metrics` 儲存已登錄報酬率序列之間的自助法比較。直到 2026-08-31，這兩張表在此都還是空的，因此較早版本的 README 引用了沒有實際資料支撐的折減數字。問題不是計算錯誤，而是本 Notebook 從未要求計算；`etfs`、`fx_pairs` 與 `us_firm_characteristics` 都有這麼做。

`compute_and_register` 會更新整張表，而非只更新單一資料列，因此不會回報過時的領先者。`populate_paired_metrics` 會為每種比較類型寫入一列，其中包括 `val_rank1_self`：選定設定的驗證期序列，與其自身保留樣本重播結果的比較。這是驗證期與保留樣本問題的配對形式，也是唯一誠實的問法。直接比較兩個點估計值，問的不是這個問題：保留樣本期間較短，因此兩者之差包含點估計值無法呈現的抽樣誤差。

選定設定在此解析，而非留到下方才解析，因為 `populate_paired_metrics` 需要它。若省略並不會失敗，而是退回依原始夏普比率為登錄資料庫排序；在此資料庫中，這會選出 `latent_factors`／`sdf` 在 `fwd_ret_21d` 上的結果，而正式解析器選出的是 `gbm`／`leaves_31_mse` 在 `fwd_ret_5d` 上的結果。配對資料列隨後會比較本章並未報告的策略，表格標題卻宣稱描述的是本章策略。這與下方記錄的保留樣本查找不一致屬於同一問題，只是影響到另一張表。
<!-- ML4T_ZH_END c0007-p000 -->

`replace_all=True` makes the call a snapshot rather than an insert. Registration is an upsert
keyed on the pair, so it cannot remove rows a previous selection wrote; without the prune, the
raw-Sharpe pairs would survive alongside the selected configuration's.

`prediction_hashes` scopes the cohorts to this notebook's own pool. On this registry it changes
nothing - the cohorts are already a strict subset of the pool, because it was rebuilt from empty
and holds no retired generation. That is a property of the registry, not of the call: without the
argument, a superseded generation left in the registry would inflate K and could lead a cohort
outright, and the deflation this notebook publishes would be computed over a variant the pool
excludes. Being right by accident is not the same as being right.

<!-- ML4T_ZH_BEGIN c0007-p001 -->
`replace_all=True` 使這次呼叫產生快照，而非只新增資料。登錄採用以配對為鍵值的 upsert，因此無法移除先前選擇寫入的資料列；若不清除，依原始夏普比率選出的配對，會與選定設定的配對一起留下。

`prediction_hashes` 將群組範圍限定在本 Notebook 自身的候選池。在目前的登錄資料庫中，這不會改變結果：群組本已嚴格屬於候選池的子集，因為資料庫是從空白狀態重建，沒有已汰換版本。但這是登錄資料庫的特性，不是呼叫本身的保證。若沒有這個參數，資料庫中留下的已汰換版本會膨脹 K 值，甚至可能成為群組領先者，使本 Notebook 發布的折減結果根據候選池排除的變體計算。碰巧得到正確結果，不等於方法正確。
<!-- ML4T_ZH_END c0007-p001 -->


In [ ]:
carrier = resolve_solvent_carrier("cme_futures")
cohort_counts = compute_and_register(
    "cme_futures",
    prediction_hashes=pool.get_column("prediction_hash").unique().to_list(),
    verbose=False,
)
# The cohort call above is scoped to the reported pool and this one is not: the pairs are
# selected from every registered prediction set. Stated rather than defaulted, and
# narrowing that scope is a separate decision from this line.
paired_rows = populate_paired_metrics(
    "cme_futures",
    carrier=carrier,
    replace_all=True,
    prediction_hashes=ENTIRE_REGISTRY,
    verbose=False,
)
print(f"cohort_metrics: {sum(cohort_counts[k] for k in ('family', 'stagelabel', 'label'))} rows")
print(f"backtest_paired_metrics: {sum(1 for r in paired_rows if 'skip' not in r)} pairs")

In [ ]:
pool_size

## What each selection stage contributed

The three stages run in sequence, each on the survivors of the one before. The baseline stage
carries every configuration and checkpoint at equal weight. Allocation runs on the strongest
distinct configurations from that stage, and the risk overlay on the strongest result so far for
each horizon. Later stages therefore hold far fewer candidates than the first, and the spread
within a stage shows how much of the outcome the sizing and risk rules decide once the model is
fixed.

**The medians cannot be read across stages.** Each stage runs on the survivors of the one before,
chosen on the same validation Sharpe the table reports, so the pool shrinks from 496 to 60 to 14
by selecting on the quantity being summarized. On `fwd_ret_21d` the median rises from -0.392 to
0.192 to 1.010 along that shrinking pool, and almost all of that movement is the selection, not
the position sizing methods or the risk rules. What the stages do support is the comparison
within a row: the fourteen risk overlays share one model, one signal and one sizing rule,
and they still span 0.322 to 1.274, which is the range the position rule alone is
responsible for.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
## 各選擇階段的貢獻

三個階段依序執行，每個階段都使用前一階段留下的候選者。基準階段以等權重納入所有設定與檢查點。配置階段處理其中最強的不同設定，風險覆蓋層則處理每個預測期間到目前為止最強的結果。因此，後續階段的候選者遠少於第一階段；單一階段內的差距，顯示模型固定後，部位規模與風險規則對結果有多大影響。

**不能跨階段比較中位數。** 每個階段都使用前一階段的勝出者，而這些勝出者是根據表格所報告的同一個驗證期夏普比率選出的。因此，候選池從 496 縮小至 60，再縮小至 14，縮減時正是依被摘要的數值做選擇。在 `fwd_ret_21d` 上，中位數隨候選池縮小由 -0.392 升至 0.192，再升至 1.010；幾乎所有變化都來自選擇，而非部位規模決定方法或風險規則。各階段真正支持的是同一資料列內的比較：十四種風險覆蓋層共用同一模型、訊號與部位規模規則，結果仍分布在 0.322 至 1.274 之間；這個範圍才是部位規則本身造成的。
<!-- ML4T_ZH_END c0010-p000 -->


In [ ]:
stage_summary = (
    pool.group_by("label", "stage")
    .agg(
        pl.len().alias("candidates"),
        pl.col("sharpe").min().alias("min_sharpe"),
        pl.col("sharpe").median().alias("median_sharpe"),
        pl.col("sharpe").max().alias("max_sharpe"),
    )
    .sort("label", "stage")
)

In [ ]:
stage_summary

In [ ]:
fig = px.strip(
    pool.to_pandas(),
    x="stage",
    y="sharpe",
    color="label",
    stripmode="overlay",
    category_orders={"stage": ["signal", "allocation", "risk_overlay"]},
    color_discrete_sequence=[COLORS["blue"], COLORS["amber"]],
    labels={"stage": "Selection stage", "sharpe": "Validation Sharpe", "label": "Return horizon"},
)
fig.update_layout(
    title="Validation Sharpe by selection stage and return horizon",
    height=420,
)
fig.add_hline(y=0, line_dash="dash", line_color=COLORS["neutral"])
fig.show()

## The selected configuration

The selection is made by `resolve_solvent_carrier`, the shared resolver, and not by ranking this
pool's Sharpe column directly. The two do not agree here. Ranking the column names the
`latent_factors` / `sdf` row on `fwd_ret_21d` at 1.274; the resolver names the `gbm` /
`leaves_31_mse` row on `fwd_ret_5d`, whose raw 1.236 becomes 1.294 once the candidates are
compared over the 1,270 sessions they all price. Different family, different horizon, from the
same registry.

The re-ranking is the reason to prefer the resolver. A Sharpe computed over a configuration's own
available history is not comparable across configurations that priced different spans, and
ranking the raw column silently rewards whichever candidate had the most forgiving window. The
resolver also refuses a selected configuration that is insolvent rather than reporting it.

It matters here beyond correctness of the ranking. `17_holdout_predictions` and
`18_holdout_backtest` resolve it the same way, so a second selection rule
in this notebook would ask `select_holdout_self_backtest` for the holdout replay of a
configuration those notebooks never ran. The answer would be `None`, and this notebook would
report the holdout as not produced while it sat in the registry.

The prediction checkpoint is part of the identity either way: two rows from the same trained
model at different checkpoints are different configurations, and a holdout matched on the
trained model alone can land on a different checkpoint from the one selected.

<!-- ML4T_ZH_BEGIN c0014-p000 -->
## 選定的設定

選擇由共用解析器 `resolve_solvent_carrier` 完成，而不是直接對這個候選池的夏普比率欄位排序。兩者在此得到不同結果。直接排序欄位會選出 `latent_factors`／`sdf` 在 `fwd_ret_21d` 上、夏普比率為 1.274 的資料列；解析器則選出 `gbm`／`leaves_31_mse` 在 `fwd_ret_5d` 上的資料列。後者的原始值為 1.236，但在所有候選者都可計價的 1,270 個交易時段上比較後，變成 1.294。同一個登錄資料庫，卻得到不同的模型家族與預測期間。

重新排序是應採用解析器的原因。根據各設定自身可用的歷史資料計算出的夏普比率，無法直接比較計價期間不同的設定；直接依原始欄位排序，會悄悄獎勵碰巧擁有較有利期間的候選者。解析器也會拒絕無償付能力的選定設定，而非照常報告。

這點的影響不止於排序正確與否。`17_holdout_predictions` 與 `18_holdout_backtest` 都用相同方式解析設定，因此若本 Notebook 採用第二套選擇規則，就會要求 `select_holdout_self_backtest` 查找那些 Notebook 從未執行過的設定之保留樣本重播結果。答案會是 `None`，於是本 Notebook 會報告保留樣本尚未產生，儘管它已存於登錄資料庫。

無論採哪種方式，預測檢查點都是識別資訊的一部分：同一訓練模型在不同檢查點產生的兩列，是不同設定；若只根據訓練模型配對保留樣本，可能配到與選定檢查點不同的結果。
<!-- ML4T_ZH_END c0014-p000 -->


In [ ]:
selected = next(
    (result for result in pool_results if result.hash == carrier["val_backtest_hash"]), None
)
if selected is None:
    raise RuntimeError(
        f"the resolved configuration {carrier['val_backtest_hash']} ({carrier['family']}/"
        f"{carrier['config_name']}, {carrier['label']}, stage {carrier['val_stage']}) is not in "
        "this notebook's pool. The pool and the shared resolver are reading the same registry, so "
        "they disagree about which stages are selected from, and the holdout notebooks followed "
        "the resolver."
    )
selected_row = pool.filter(pl.col("backtest_hash") == selected.hash)
selected_label = selected_row.item(0, "label")
selected_strategy = selected.spec()["strategy"]

In [ ]:
selected_row

In [ ]:
pl.DataFrame(
    [
        {
            "candidate_set_hash": pool_identity,
            "candidates_compared": len(pool_results),
            "label": selected_label,
            "signal": str(selected_strategy["signal"]),
            "allocation": str(selected_strategy.get("allocation")),
            "risk": str(selected_strategy.get("risk")),
        }
    ]
)

## What friction costs this configuration

The cost grid was run on the single configuration this case study ships - the same one selected
above, resolved across labels and priced with its risk overlay in place - holding the model,
sizing, risk rules and contract specification fixed and varying only the all-in cost assumption.
Commission and slippage each take half of the quoted figure. One curve, not one per horizon:
there is one strategy, so the label the selected configuration does not sit on has no cost rows
at all.

<!-- ML4T_ZH_BEGIN c0018-p000 -->
## 交易摩擦讓這項設定付出多少成本

成本網格只針對本案例研究最終採用的一項設定執行：也就是上方跨標籤選出，並保留風險覆蓋層以估算成本的同一項設定。模型、部位規模、風險規則與合約規格都固定，只改變全包成本假設。引用數值中的佣金與滑價各占一半。只有一條曲線，而非每個預測期間各一條：策略只有一個，因此選定設定不屬於的標籤完全沒有成本資料列。
<!-- ML4T_ZH_END c0018-p000 -->


In [ ]:
if EXECUTION_TIER == "canonical":
    cost_population = OfficialPopulation.one(study, name="cme_futures-cost-validation-v1")
    cost_members = list(cost_population.require_complete())
else:
    cost_members = (
        study.backtests.table(include_preview=True)
        .filter(
            (pl.col("execution_tier") == "preview")
            & (pl.col("stage") == "cost_sensitivity")
            & pl.col("complete")
        )
        .get_column("backtest_hash")
        .to_list()
    )
cost_curve = (
    study.backtests.table(include_preview=True)
    .filter(pl.col("backtest_hash").is_in(cost_members) & (pl.col("label") == selected_label))
    .with_columns(
        (
            pl.col("spec_json")
            .str.json_path_match("$.decision_artifact.parameters.costs.commission_bps")
            .cast(pl.Float64)
            + pl.col("spec_json")
            .str.json_path_match("$.decision_artifact.parameters.costs.slippage_bps")
            .cast(pl.Float64)
        ).alias("total_cost_bps")
    )
    .select("total_cost_bps", "sharpe", "total_return", "num_trades", "backtest_hash")
    .sort("total_cost_bps")
)
if cost_curve.is_empty():
    raise RuntimeError(f"the cost population contains no member for {selected_label!r}")
# `json_path_match` returns null for a path that is not in the document rather than raising, and
# null + null is null, so reading the grid value from the wrong place yields a full-height frame
# whose cost axis is entirely missing. The emptiness check above passes on such a frame and the
# curve below plots against nothing. Refuse it here instead.
if cost_curve.get_column("total_cost_bps").null_count():
    raise RuntimeError(
        "cost members record no all-in cost at "
        "$.decision_artifact.parameters.costs; the cost curve has no axis"
    )

In [ ]:
cost_curve

In [ ]:
fig = px.line(
    cost_curve.to_pandas(),
    x="total_cost_bps",
    y="sharpe",
    markers=True,
    labels={"total_cost_bps": "All-in cost (bps per trade)", "sharpe": "Validation Sharpe"},
    color_discrete_sequence=[COLORS["copper"]],
)
fig.update_layout(
    title="Validation Sharpe across the all-in transaction-cost grid",
    height=380,
)
fig.add_hline(y=0, line_dash="dash", line_color=COLORS["neutral"])
fig.show()

## The holdout

The holdout evaluates one configuration: the one the validation backtests selected. That is the
highest validation backtest Sharpe across the baseline, position-sizing, allocation and
risk-management stages, and it is fixed before any holdout artifact exists.

What keeps the holdout from becoming an axis to search over is the direction of that rule, not a
gate. The ranking reads validation rows only, and the holdout row below is found by matching the
selected strategy specification - never by taking whichever holdout backtest scored best. A
holdout number therefore cannot change which configuration is reported here.

Nothing about it is one-shot. A holdout result that turns out to be wrong is deleted and produced
again; what would make the number uninterpretable is evaluating many configurations on the window
and reporting the best, which is the thing the selection rule rules out. The holdout notebooks
produce the row; this one reads it. Where they have not run, the table is empty and the validation
result above stands on its own.

<!-- ML4T_ZH_BEGIN c0022-p000 -->
## 保留樣本

保留樣本只評估一項設定：驗證期回測選出的設定。它是基準、部位規模、配置與風險管理階段中，驗證期回測夏普比率最高的設定，並在任何保留樣本產出檔存在前就已固定。

防止保留樣本變成可供搜尋的另一個維度，靠的是這條規則的方向，而非一道關卡。排序只讀取驗證期資料列；下方的保留樣本資料列，則依與選定策略規格相符的條件查找，絕不選取保留樣本回測中分數最高者。因此，保留樣本數字無法改變此處報告的設定。

這並非只能執行一次。若保留樣本結果後來證實有誤，可以刪除並重新產生；真正讓數字無法解讀的，是在這個期間評估多項設定，再只報告最佳者，而選擇規則已排除這種做法。保留樣本 Notebook 會產生資料列，本 Notebook 則讀取它。若保留樣本 Notebook 尚未執行，表格就是空的，上方的驗證期結果仍可獨立呈現。
<!-- ML4T_ZH_END c0022-p000 -->


In [ ]:
# `select_holdout_self_backtest` is the shared resolver every strategy-analysis notebook uses.
# It takes the selection this notebook already made and finds the holdout backtest replaying that
# same strategy specification, at the same configuration and checkpoint, over a training run whose
# own CV declares the holdout fold. It returns None where no such run exists, and raises rather
# than choosing where two of them do.
#
# Calling it rather than re-deriving the lineage here is deliberate. A second implementation
# living beside the first agrees with it on the registry it was written against and diverges on
# the next one, and a divergence in this particular lookup is a holdout number attributed to the
# wrong configuration.
holdout_backtest_hash = select_holdout_self_backtest("cme_futures", selected.hash)
print(
    f"Selected validation backtest: {selected.hash}  ({selected_label})\n"
    f"Holdout replay: {holdout_backtest_hash or 'not produced yet'}"
)

In [ ]:
if holdout_backtest_hash is None:
    comparison = pl.DataFrame()
else:
    evaluated = study.backtests.table(include_preview=True).filter(
        pl.col("backtest_hash") == holdout_backtest_hash
    )
    comparison = pl.concat(
        [
            selected_row.select("label", "sharpe", "max_drawdown", "num_trades").with_columns(
                pl.lit("validation").alias("split")
            ),
            evaluated.select("label", "sharpe", "max_drawdown", "num_trades").with_columns(
                pl.lit("holdout").alias("split")
            ),
        ]
    ).select("split", "label", "sharpe", "max_drawdown", "num_trades")
comparison